# MENTORX: AKT vs DKT-Q on FoundationalASSIST

**Question:** does **AKT** (Attentive Knowledge Tracing, Ghosh, Heffernan & Lan, KDD 2020) predict
whether a student answers the next maths question correctly better than **DKT-Q**, the model
MENTORX serves today?

**Fair comparison:**
- Same cleaning and the same **student split (70/15/15, seed 42)** as
  `02_foundationalassist_knowledge_tracing` (the notebook that trained the deployed DKT-Q).
- Same label: 1 = correct on the first attempt with no hint and the answer not shown.
- Same extra inputs (hints, answer shown, time gap of *earlier* attempts) and the same
  first-attempt fallback (question success rate).
- DKT-Q is **retrained in this run** on the same data, so both models face the same test students.
  The paired bootstrap at the end gives a confidence interval for the AUC difference.

**Round 1 of this notebook (Kaggle T4, `ml/notebooks/akt_outputs/run1/`):** AKT test AUC 0.8103 vs
DKT-Q 0.8118 (paired difference −0.0015, CI −0.0021 to −0.0007), but AKT was better calibrated
(ECE 0.0078 vs 0.0148) and was **still improving at its last epoch** (30 of 30, 37 s each).
**Round 2 (this version)** trains three AKT settings to convergence (up to 200 epochs, early stopping),
picks the best one on validation only, and then scores it against DKT-Q on the test students once.

**Results of the earlier run (from `ml/notebooks/kt_outputs/results.json`, test students):**
DKT-Q AUC 0.8117 (CI 0.806–0.818), LogLoss 0.5085. Transformer-KT AUC 0.8070.
Gradient boosting AUC 0.8227.

**How to run on Kaggle**
1. Create a new notebook (or *File → Import notebook*) and upload this file.
2. *Add data* → attach the **foundational-assist-dataset** (the one with `Interactions.csv`,
   `Problems.csv`, `Skills.csv`).
3. *Settings → Accelerator → GPU T4 x1* (or P100). *Settings → Internet* can stay off.
4. Click **Save Version → Save & Run All (Commit)**. You can close the browser; the run continues.
5. When it finishes, open the version → *Output*, and download the `mentorx_akt` folder.
   Send back what the last cell prints.

If the session is cut off, run it again: finished models are loaded from `/kaggle/working`
only within the same session, so it's best to commit and let it finish.

In [ ]:
import copy
import gc
import json
import math
import os
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42


def set_seed(seed: int = SEED) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = os.environ.get("MENTORX_SMOKE") == "1"  # tiny local test only; never set on Kaggle

# Data: Kaggle mounts datasets under /kaggle/input; MENTORX_DATA_DIR overrides it for a local run
DATA_DIR = Path(os.environ.get("MENTORX_DATA_DIR", "/kaggle/input"))
if not (DATA_DIR / "Interactions.csv").exists():
    found = sorted(DATA_DIR.rglob("Interactions.csv")) if DATA_DIR.exists() else []
    if not found:
        raise FileNotFoundError("Attach the foundational-assist-dataset to this notebook (Add data).")
    DATA_DIR = found[0].parent
OUT_DIR = Path(os.environ.get("MENTORX_OUT_DIR", "/kaggle/working/mentorx_akt"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Settings ----
MAX_EPOCHS = 1 if SMOKE else 200  # early stopping ends runs well before this
PATIENCE = 5                      # epochs without a better validation log loss
# Kaggle stops a session at 12 h; each run keeps its best epoch if it hits its limit
MAX_HOURS = {"AKT-A": 3.0, "AKT-B": 2.5, "AKT-C": 3.5, "DKT-Q": 1.0}
TRAIN_DKTQ_BASELINE = True        # retrain DKT-Q on the same split for a paired comparison

AKT_BASE = dict(dim=128, heads=8, n_blocks=2, d_ff=512, dropout=0.2,
                use_feats=True)  # use_feats: hints / answer shown / time gap of earlier attempts, like DKT-Q
# Three settings; the best one is chosen on VALIDATION log loss only
AKT_RUNS = {
    "AKT-A": dict(cfg=AKT_BASE, lr=1e-4),                                 # round 1, trained to convergence
    "AKT-B": dict(cfg=AKT_BASE, lr=3e-4),                                 # faster learning rate
    "AKT-C": dict(cfg={**AKT_BASE, "dim": 256, "d_ff": 1024}, lr=2e-4),   # the paper's width
}
if SMOKE:
    AKT_RUNS = {"AKT-A": dict(cfg=AKT_BASE, lr=1e-3), "AKT-B": dict(cfg={**AKT_BASE, "dim": 64}, lr=1e-3)}
L2_RASCH = 1e-5       # L2 on the per-question difficulty (as in the AKT paper)
WEIGHT_DECAY = 1e-5
MAX_BATCH = 64
ATTN_BUDGET = 4_000_000   # batch_size * seq_len^2 limit; keeps attention memory well under 16 GB
TRAIN_WINDOW = 1000       # long training sequences are cut into windows of at most this many attempts
EVAL_WINDOW = 1000        # longer test sequences are scored with sliding windows (>= 500 attempts of context)

if DEVICE.type == "cpu" and not SMOKE:
    print("WARNING: no GPU. Settings -> Accelerator -> GPU T4, or this will take many hours.")
print("Data:", DATA_DIR)
print("Outputs:", OUT_DIR.resolve())
print("Device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
      "| torch", torch.__version__, "| smoke" if SMOKE else "")

## 1. Load and clean (identical to notebook 02)

Duplicate interaction ids removed, rows without a score dropped, mixed timestamp formats parsed,
sorted by student, time and id. Duplicate `problem_id` rows in Problems and duplicate
(problem, skill) pairs in Skills are dropped.

In [ ]:
raw = pd.read_csv(DATA_DIR / "Interactions.csv", index_col=0)
problems_raw = pd.read_csv(DATA_DIR / "Problems.csv")
skills_raw = pd.read_csv(DATA_DIR / "Skills.csv")

df = raw.drop_duplicates(subset="id").dropna(subset=["discrete_score"]).copy()
df["end_time"] = pd.to_datetime(df["end_time"], utc=True, format="ISO8601", errors="coerce")
df = df.dropna(subset=["end_time"])
df["correct"] = df["discrete_score"].astype(np.int8)
df = df.sort_values(["user_id", "end_time", "id"]).reset_index(drop=True)
problems = problems_raw.drop_duplicates(subset="problem_id").reset_index(drop=True)
skills = skills_raw.drop_duplicates(subset=["problem_id", "skill_id"])
print(f"Interactions: {len(raw):,} raw -> {len(df):,} clean")
del raw
df = df.drop(columns=["answer_text", "discrete_score"])
gc.collect()

if SMOKE:  # keep 120 students so a CPU test finishes in minutes; the split below still uses seed 42
    keep = np.random.default_rng(0).choice(df.user_id.unique(), 120, replace=False)
    df = df[df.user_id.isin(keep)].reset_index(drop=True)

print(f"Students {df.user_id.nunique():,} | questions {df.problem_id.nunique():,} | "
      f"skills {skills.skill_id.nunique()} | correct rate {df.correct.mean():.3f}")

## 2. Index skills and questions, split by student (identical to notebook 02)

A question that no training student attempted maps to index 0 ("unknown"), so no model gets a
learned embedding for a question it could not have learned from.

In [ ]:
student_ids = np.sort(df["user_id"].unique())
train_students, temp = train_test_split(student_ids, test_size=0.30, random_state=SEED)
val_students, test_students = train_test_split(temp, test_size=0.50, random_state=SEED)
split_of = pd.Series("train", index=student_ids)
split_of[val_students] = "val"
split_of[test_students] = "test"
df["split"] = df["user_id"].map(split_of).values

skill_ids = np.sort(skills["skill_id"].unique())
skill_idx = {s: i + 1 for i, s in enumerate(skill_ids)}
NUM_SKILLS = len(skill_ids) + 1
MAX_SK = 3
prob_skills = skills.groupby("problem_id")["skill_id"].apply(lambda s: [skill_idx[x] for x in s][:MAX_SK])
sk = np.zeros((len(df), MAX_SK), dtype=np.int64)
for j, lst in enumerate(df["problem_id"].map(prob_skills)):
    if isinstance(lst, list):
        sk[j, : len(lst)] = lst

train_q = np.sort(df.loc[df.split == "train", "problem_id"].unique())
q_idx = {q: i + 1 for i, q in enumerate(train_q)}
NUM_Q = len(train_q) + 1
df["q"] = df["problem_id"].map(q_idx).fillna(0).astype(np.int64)

for name in ["train", "val", "test"]:
    part = df[df.split == name]
    print(f"{name:5s}: {part.user_id.nunique():5,} students  {len(part):9,} interactions  "
          f"unknown-question rows {(part.q == 0).mean():.2%}")

## 3. Metrics, baseline for first attempts, sequences

In [ ]:
def ece_score(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    return sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean()) for b in range(bins) if (idx == b).any())


def metrics(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    m = {"AUC": roc_auc_score(y, p), "LogLoss": log_loss(y, p, labels=[0, 1]),
         "Accuracy": accuracy_score(y, p >= 0.5), "Brier": brier_score_loss(y, p), "ECE": ece_score(y, p)}
    return {k: float(v) for k, v in m.items()}  # plain floats: checkpoints stay loadable with weights_only


val_rows = df.index[df.split == "val"].to_numpy()
test_rows = df.index[df.split == "test"].to_numpy()
y_val = df.loc[val_rows, "correct"].to_numpy()
y_test = df.loc[test_rows, "correct"].to_numpy()

# Question success rate from training students: used only for each student's very first attempt
tr = df[df.split == "train"]
glob = tr.correct.mean()
g = tr.groupby("problem_id")["correct"].agg(["sum", "count"])
q_rate = (g["sum"] + 2 * glob) / (g["count"] + 2)


def first_attempt_fill(p, rows):
    out = p[rows].copy()
    miss = np.isnan(out)
    out[miss] = df.loc[rows[miss], "problem_id"].map(q_rate).fillna(glob).to_numpy()
    return out


# Per-attempt inputs used only for LATER predictions: hints, answer shown, gap since the previous attempt
secs_all = df["end_time"].astype("int64").to_numpy() / 1e9
users = df["user_id"].to_numpy()
new_user = np.r_[True, users[1:] != users[:-1]]
gap = np.r_[0.0, np.diff(secs_all)]
gap[new_user] = 0.0
feat_all = np.stack([
    np.log1p(df["hint_count"].clip(upper=10).to_numpy()),
    df["saw_answer"].astype(float).to_numpy(),
    np.log1p(np.maximum(gap, 0)) / 10.0,
], axis=1).astype(np.float32)

USER_START = np.flatnonzero(new_user)
USER_END = np.r_[USER_START[1:], len(df)]
Q_ALL = df["q"].to_numpy().copy()
R_ALL = df["correct"].to_numpy().astype(np.int64)
SPLIT_ALL = df["split"].to_numpy()
SEQ = {s: [] for s in ["train", "val", "test"]}
for a, b in zip(USER_START, USER_END):
    SEQ[SPLIT_ALL[a]].append({"rows": np.arange(a, b), "q": Q_ALL[a:b], "sk": sk[a:b], "r": R_ALL[a:b], "f": feat_all[a:b]})
lens = np.array([len(x["r"]) for v in SEQ.values() for x in v])
MAX_LEN = int(lens.max())
print({k: len(v) for k, v in SEQ.items()}, "| attempts per student: median", int(np.median(lens)),
      "p95", int(np.percentile(lens, 95)), "max", MAX_LEN,
      f"| students longer than {EVAL_WINDOW}: {(lens > EVAL_WINDOW).sum()}")

## 4. AKT

Faithful to the paper and its official code (`arghosh/AKT`), with two adaptations for this data:

- **Rasch-model embeddings.** Question *q* with concept *c*: `x = c_emb + mu_q * d_c`, and the
  response embedding `y = (c,r)_emb + mu_q * f_(c,r)`. `mu_q` is one learned difficulty number per
  question, so similar questions share the concept embedding and differ only by difficulty.
  *Adaptation:* a question can have up to 3 skills, so concept embeddings are averaged (as DKT-Q does).
- **Monotonic attention.** Each head multiplies its attention scores by `exp(-theta * distance)`, where
  the distance is context-aware (how much attention mass lies between two attempts). Old attempts fade
  unless they are very relevant. There are no positional embeddings.
- **Three parts:** a question encoder (self-attention over questions), a knowledge encoder
  (self-attention over responses) and a knowledge retriever, where the current question attends to
  **strictly earlier** responses. The prediction is an MLP over the retrieved knowledge and the
  current question embedding.
- *Adaptation:* the hint / answer-shown / time-gap features of an attempt are added to its response
  embedding, so they are only ever seen by later predictions (same rule as DKT-Q).

Leakage check: the prediction at position *t* uses questions 0..t (no outcomes) and responses 0..t-1.

In [ ]:
class SkillBag(nn.Module):
    # Mean of up to 3 skill embeddings (index 0 = padding)
    def __init__(self, n, dim):
        super().__init__()
        self.emb = nn.Embedding(n, dim, padding_idx=0)

    def forward(self, idx):
        e = self.emb(idx)
        return e.sum(-2) / (idx > 0).sum(-1, keepdim=True).clamp(min=1)


def monotonic_attention(q, k, v, mask, gamma, dropout, zero_pad):
    # q, k, v: (B, H, T, dk); mask: (T, T) bool, True = may attend; gamma: (H, 1, 1)
    dk = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(dk)
    T = scores.size(-1)
    pos = torch.arange(T, device=q.device)
    position_effect = (pos[None, :] - pos[:, None]).abs().float()[None, None]
    with torch.no_grad():
        s_ = scores.masked_fill(~mask, -1e32).softmax(-1) * mask.float()
        distcum = s_.cumsum(-1)
        disttotal = s_.sum(-1, keepdim=True)
        dist = ((disttotal - distcum) * position_effect).clamp(min=0).sqrt()
    theta = -F.softplus(gamma).unsqueeze(0)  # (1, H, 1, 1), always negative
    total_effect = (dist * theta).exp().clamp(min=1e-5, max=1e5)
    scores = (scores * total_effect).masked_fill(~mask, -1e32).softmax(-1)
    if zero_pad:  # first position has no earlier response to attend to
        scores = torch.cat([torch.zeros_like(scores[:, :, :1]), scores[:, :, 1:]], dim=2)
    return dropout(scores) @ v


class MonoMHA(nn.Module):
    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.h, self.dk = heads, dim // heads
        self.kq = nn.Linear(dim, dim)  # keys and queries share weights (kq_same in the paper)
        self.v = nn.Linear(dim, dim)
        self.out = nn.Linear(dim, dim)
        self.gamma = nn.Parameter(torch.zeros(heads, 1, 1))
        nn.init.xavier_uniform_(self.gamma)
        self.drop = nn.Dropout(dropout)

    def forward(self, q, k, v, strict):
        B, T, D = q.shape
        split = lambda t: t.view(B, T, self.h, self.dk).transpose(1, 2)
        mask = torch.ones(T, T, dtype=torch.bool, device=q.device).tril(-1 if strict else 0)
        o = monotonic_attention(split(self.kq(q)), split(self.kq(k)), split(self.v(v)), mask, self.gamma,
                                self.drop, zero_pad=strict)
        return self.out(o.transpose(1, 2).reshape(B, T, D))


class AKTBlock(nn.Module):
    def __init__(self, dim, heads, d_ff, dropout):
        super().__init__()
        self.attn = MonoMHA(dim, heads, dropout)
        self.n1, self.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.d1, self.d2 = nn.Dropout(dropout), nn.Dropout(dropout)
        self.ff = nn.Sequential(nn.Linear(dim, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, dim))

    def forward(self, q, k, v, strict, apply_ff=True):
        x = self.n1(q + self.d1(self.attn(q, k, v, strict)))
        if apply_ff:
            x = self.n2(x + self.d2(self.ff(x)))
        return x


class AKT(nn.Module):
    def __init__(self, dim=128, heads=8, n_blocks=2, d_ff=512, dropout=0.2, use_feats=True):
        super().__init__()
        self.c = SkillBag(NUM_SKILLS, dim)          # concept
        self.d = SkillBag(NUM_SKILLS, dim)          # concept variation
        self.cr = SkillBag(2 * NUM_SKILLS, dim)     # (concept, response)
        self.fr = SkillBag(2 * NUM_SKILLS, dim)     # (concept, response) variation
        self.mu = nn.Embedding(NUM_Q, 1, padding_idx=0)  # Rasch difficulty per question (0 = unknown)
        nn.init.zeros_(self.mu.weight)
        self.feat = nn.Linear(3, dim) if use_feats else None
        self.y_blocks = nn.ModuleList(AKTBlock(dim, heads, d_ff, dropout) for _ in range(n_blocks))
        self.x_blocks = nn.ModuleList(AKTBlock(dim, heads, d_ff, dropout) for _ in range(2 * n_blocks))
        self.head = nn.Sequential(
            nn.Linear(2 * dim, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))

    def forward(self, b):
        q, s, r = b["q"], b["sk"], b["r"]
        mu = self.mu(q)                                       # (B, T, 1)
        x = self.c(s) + mu * self.d(s)                        # question embedding (no outcome)
        sr = torch.where(s > 0, 2 * s + r.unsqueeze(-1), torch.zeros_like(s))
        y = self.cr(sr) + mu * self.fr(sr)                    # response embedding
        if self.feat is not None:
            y = y + self.feat(b["f"])
        for blk in self.y_blocks:                             # knowledge encoder (responses 0..t)
            y = blk(y, y, y, strict=False)
        h = x
        for i, blk in enumerate(self.x_blocks):
            if i % 2 == 0:                                    # question encoder (questions 0..t)
                h = blk(h, h, h, strict=False, apply_ff=False)
            else:                                             # retriever: responses 0..t-1 only
                h = blk(h, h, y, strict=True)
        logit = self.head(torch.cat([h, x], -1)).squeeze(-1)
        return logit, mu.squeeze(-1)


# Leakage test: changing the response at position t must not change the prediction at t (or before)
_m = AKT(**{**AKT_BASE, "dim": 32, "heads": 4}).eval()
_b = {"q": torch.randint(1, NUM_Q, (1, 12)), "sk": torch.randint(1, NUM_SKILLS, (1, 12, MAX_SK)),
      "r": torch.randint(0, 2, (1, 12)), "f": torch.rand(1, 12, 3)}
_b2 = {**_b, "r": _b["r"].clone(), "f": _b["f"].clone()}
_b2["r"][0, 7] = 1 - _b2["r"][0, 7]
_b2["f"][0, 7] += 1
with torch.no_grad():
    _p1, _ = _m(_b)
    _p2, _ = _m(_b2)
assert torch.allclose(_p1[0, :8], _p2[0, :8], atol=1e-5), "AKT sees the answer it is predicting"
assert not torch.allclose(_p1[0, 8:], _p2[0, 8:]), "later predictions should use that answer"
print("Leakage test passed.")
del _m, _b, _b2

## 5. Batching and training

Batches are grouped by length so that `batch × length²` stays under `ATTN_BUDGET` (long students get
small batches, short students large ones). Very long training sequences are cut into windows of
`TRAIN_WINDOW` attempts. The first attempt of each window is not scored, since it has no history inside
the window. Training: AdamW, learning rate halved on a validation plateau, early stopping on
validation log loss, gradient clipping. The best model is written to disk after every improvement.

In [ ]:
def make_items(split, window, train):
    # (sequence, start, end, score_from): positions [score_from, end) of the window are scored
    items = []
    for x in SEQ[split]:
        n = len(x["r"])
        if n < 2:
            continue
        if n <= window:
            items.append((x, 0, n, 1))
        elif train:
            for a in range(0, n - 1, window):
                items.append((x, a, min(a + window, n), a + 1))
        else:  # sliding windows; each scored attempt has at least window/2 earlier attempts in view
            items.append((x, 0, window, 1))
            s, step = window, window // 2
            while s < n:
                e = min(s + step, n)
                items.append((x, e - window, e, s))
                s = e
    return items


def batches(items, shuffle):
    # Sorted by length, so the newest item is always the longest in its group
    order = np.argsort([e - a for _, a, e, _ in items], kind="stable")
    groups, cur = [], []
    for i in order:
        L = items[i][2] - items[i][1]
        if cur and ((len(cur) + 1) * L * L > ATTN_BUDGET or len(cur) >= MAX_BATCH):
            groups.append(cur)
            cur = []
        cur.append(i)
    if cur:
        groups.append(cur)
    if shuffle:
        np.random.shuffle(groups)
    return groups


def collate_full(group_items):
    T = max(e - a for _, a, e, _ in group_items)
    B = len(group_items)
    out = {"q": torch.zeros(B, T, dtype=torch.long), "sk": torch.zeros(B, T, MAX_SK, dtype=torch.long),
           "r": torch.zeros(B, T, dtype=torch.long), "f": torch.zeros(B, T, 3),
           "score": torch.zeros(B, T, dtype=torch.bool)}
    for i, (x, a, e, s0) in enumerate(group_items):
        n = e - a
        out["q"][i, :n] = torch.from_numpy(x["q"][a:e])
        out["sk"][i, :n] = torch.from_numpy(x["sk"][a:e])
        out["r"][i, :n] = torch.from_numpy(x["r"][a:e])
        out["f"][i, :n] = torch.from_numpy(x["f"][a:e])
        out["score"][i, s0 - a:n] = True
    return out


def to_device(b):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}


@torch.no_grad()
def predict_akt(model, split):
    model.eval()
    probs = np.full(len(df), np.nan)
    items = make_items(split, EVAL_WINDOW, train=False)
    for grp in batches(items, shuffle=False):
        its = [items[i] for i in grp]
        logit, _ = model(to_device(collate_full(its)))
        p = torch.sigmoid(logit).cpu().numpy()
        for i, (x, a, e, s0) in enumerate(its):
            probs[x["rows"][s0:e]] = p[i, s0 - a:e - a]
    return probs


def ckpt_path(name):
    return OUT_DIR / f"{name.lower().replace('-', '_')}.pt"


def fit(name, model, train_step, predict_fn, lr, epochs_items=None):
    # Generic loop: train_step(model, opt) -> (loss_sum, n); predict_fn(model, split) -> probs per df row
    path = ckpt_path(name)
    model = model.to(DEVICE)
    if path.exists() and os.environ.get("RETRAIN") != "1":
        ck = torch.load(path, map_location=DEVICE)
        assert (ck["num_skills"], ck["num_q"]) == (NUM_SKILLS, NUM_Q), "checkpoint is from different data"
        model.load_state_dict(ck["state_dict"])
        hist_file = path.with_suffix(".history.csv")
        print(f"{name}: loaded {path.name}")
        return model, (pd.read_csv(hist_file) if hist_file.exists() else pd.DataFrame())
    set_seed()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=1)
    vr = np.flatnonzero(SPLIT_ALL == "val")
    yv = R_ALL[vr]
    print(f"{name}: {sum(p.numel() for p in model.parameters()):,} parameters")
    best, bad, hist = math.inf, 0, []
    started = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t0 = time.time()
        tot, cnt = train_step(model, opt)
        p = predict_fn(model, "val")[vr]
        keep = ~np.isnan(p)
        m = metrics(yv[keep], p[keep])
        sched.step(m["LogLoss"])
        hist.append({"epoch": epoch, "train_loss": tot / cnt, **{f"val_{k}": v for k, v in m.items()},
                     "seconds": round(time.time() - t0)})
        line = (f"{name} epoch {epoch:2d} | train {tot / cnt:.4f} | val logloss {m['LogLoss']:.4f} "
                f"AUC {m['AUC']:.4f} acc {m['Accuracy']:.4f} | lr {opt.param_groups[0]['lr']:.0e} | "
                f"{time.time() - t0:.0f}s")
        print(line, flush=True)
        with open(OUT_DIR / "train_log.txt", "a") as fh:
            fh.write(line + "\n")
        pd.DataFrame(hist).to_csv(path.with_suffix(".history.csv"), index=False)
        if m["LogLoss"] < best - 1e-4:
            best, bad = m["LogLoss"], 0
            torch.save({"state_dict": copy.deepcopy(model.state_dict()), "num_skills": NUM_SKILLS, "num_q": NUM_Q,
                        "epoch": epoch, "val": m}, path)
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  early stop")
                break
        if time.time() - started > MAX_HOURS.get(name, 99) * 3600:
            print(f"  time limit ({MAX_HOURS[name]} h) reached; keeping the best epoch so far")
            break
    model.load_state_dict(torch.load(path, map_location=DEVICE)["state_dict"])
    return model, pd.DataFrame(hist)


TRAIN_ITEMS = make_items("train", TRAIN_WINDOW, train=True)


def akt_train_step(model, opt):
    tot, cnt = 0.0, 0
    for grp in batches(TRAIN_ITEMS, shuffle=True):
        b = to_device(collate_full([TRAIN_ITEMS[i] for i in grp]))
        logit, mu = model(b)
        m = b["score"]
        n = m.sum()
        loss_sum = F.binary_cross_entropy_with_logits(logit[m], b["r"][m].float(), reduction="sum")
        loss = (loss_sum + L2_RASCH * (mu[m] ** 2).sum()) / n
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot += loss_sum.item()
        cnt += n.item()
    return tot, cnt


print(f"training windows: {len(TRAIN_ITEMS):,} in {len(batches(TRAIN_ITEMS, False)):,} batches")

In [ ]:
VAL_PRED, TEST_PRED, AKT_HISTS = {}, {}, {}
for name, run in AKT_RUNS.items():
    model, AKT_HISTS[name] = fit(name, AKT(**run["cfg"]), akt_train_step, predict_akt, run["lr"])
    VAL_PRED[name] = first_attempt_fill(predict_akt(model, "val"), val_rows)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

BEST_AKT = min(AKT_RUNS, key=lambda m: metrics(y_val, VAL_PRED[m])["LogLoss"])
AKT_MODEL = AKT(**AKT_RUNS[BEST_AKT]["cfg"]).to(DEVICE)
AKT_MODEL.load_state_dict(torch.load(ckpt_path(BEST_AKT), map_location=DEVICE)["state_dict"])
AKT_HIST = AKT_HISTS[BEST_AKT]
print("Best AKT setting on validation:", BEST_AKT, AKT_RUNS[BEST_AKT])
display(pd.DataFrame({m: metrics(y_val, VAL_PRED[m]) for m in AKT_RUNS}).T.round(4))

## 6. DKT-Q baseline, retrained on the same split

The same architecture and settings as the deployed DKT-Q (notebook 02): an LSTM over
(skill, response) and (question, response) embeddings plus the earlier attempt's features, and a head
with question embedding and a per-question difficulty bias.

In [ ]:
class DKTQ(nn.Module):
    def __init__(self, dim=128, dropout=0.2):
        super().__init__()
        self.skill_resp = SkillBag(2 * NUM_SKILLS, dim)
        self.skill_out = SkillBag(NUM_SKILLS, dim)
        self.q_resp = nn.Embedding(2 * NUM_Q, dim, padding_idx=0)
        self.q_out = nn.Embedding(NUM_Q, dim, padding_idx=0)
        self.q_bias = nn.Embedding(NUM_Q, 1, padding_idx=0)
        nn.init.zeros_(self.q_bias.weight)
        self.inp = nn.Sequential(nn.Linear(2 * dim + 3, dim), nn.LayerNorm(dim), nn.Dropout(dropout))
        self.enc = nn.LSTM(dim, dim, batch_first=True)
        self.head = nn.Sequential(nn.Linear(3 * dim, dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(dim, 1))

    def forward(self, b):
        sk_in, r_in = b["sk_in"], b["r_in"]
        sr = torch.where(sk_in > 0, 2 * sk_in + r_in.unsqueeze(-1), torch.zeros_like(sk_in))
        qr = torch.where(b["q_in"] > 0, 2 * b["q_in"] + r_in, torch.zeros_like(r_in))
        x = self.inp(torch.cat([self.skill_resp(sr), self.q_resp(qr), b["f_in"]], -1))
        L = x.size(1)
        packed = nn.utils.rnn.pack_padded_sequence(x, b["lengths"].cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.enc(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=L)
        logit = self.head(torch.cat([h, self.skill_out(b["sk_out"]), self.q_out(b["q_out"])], -1)).squeeze(-1)
        return logit + self.q_bias(b["q_out"]).squeeze(-1)


def collate_shift(batch):
    L = max(len(x["r"]) for x in batch) - 1
    B = len(batch)
    out = {"q_in": torch.zeros(B, L, dtype=torch.long), "sk_in": torch.zeros(B, L, MAX_SK, dtype=torch.long),
           "r_in": torch.zeros(B, L, dtype=torch.long), "f_in": torch.zeros(B, L, 3),
           "q_out": torch.zeros(B, L, dtype=torch.long), "sk_out": torch.zeros(B, L, MAX_SK, dtype=torch.long),
           "y": torch.zeros(B, L), "mask": torch.zeros(B, L, dtype=torch.bool), "lengths": torch.zeros(B, dtype=torch.long)}
    for i, x in enumerate(batch):
        n = len(x["r"]) - 1
        out["q_in"][i, :n] = torch.from_numpy(x["q"][:-1])
        out["sk_in"][i, :n] = torch.from_numpy(x["sk"][:-1])
        out["r_in"][i, :n] = torch.from_numpy(x["r"][:-1])
        out["f_in"][i, :n] = torch.from_numpy(x["f"][:-1])
        out["q_out"][i, :n] = torch.from_numpy(x["q"][1:])
        out["sk_out"][i, :n] = torch.from_numpy(x["sk"][1:])
        out["y"][i, :n] = torch.from_numpy(x["r"][1:].astype(np.float32))
        out["mask"][i, :n] = True
        out["lengths"][i] = n
    return out


DKTQ_SEQ = {s: [x for x in SEQ[s] if len(x["r"]) >= 2] for s in SEQ}


def dktq_train_step(model, opt):
    tot, cnt = 0.0, 0
    dl = torch.utils.data.DataLoader(DKTQ_SEQ["train"], batch_size=64, shuffle=True, collate_fn=collate_shift)
    for b in dl:
        b = to_device(b)
        m = b["mask"]
        loss_sum = F.binary_cross_entropy_with_logits(model(b)[m], b["y"][m], reduction="sum")
        opt.zero_grad(set_to_none=True)
        (loss_sum / m.sum()).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot += loss_sum.item()
        cnt += m.sum().item()
    return tot, cnt


@torch.no_grad()
def predict_dktq(model, split):
    model.eval()
    probs = np.full(len(df), np.nan)
    seqs = DKTQ_SEQ[split]
    for i in range(0, len(seqs), 64):
        chunk = seqs[i:i + 64]
        p = torch.sigmoid(model(to_device(collate_shift(chunk)))).cpu().numpy()
        for j, x in enumerate(chunk):
            probs[x["rows"][1:]] = p[j, : len(x["r"]) - 1]
    return probs


if TRAIN_DKTQ_BASELINE:
    DKTQ_MODEL, DKTQ_HIST = fit("DKT-Q", DKTQ(), dktq_train_step, predict_dktq, lr=1e-3)
    VAL_PRED["DKT-Q"] = first_attempt_fill(predict_dktq(DKTQ_MODEL, "val"), val_rows)

val_table = pd.DataFrame({m: metrics(y_val, VAL_PRED[m]) for m in VAL_PRED}).T.sort_values("LogLoss")
display(val_table.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h in list(AKT_HISTS.items()) + ([("DKT-Q", DKTQ_HIST)] if TRAIN_DKTQ_BASELINE else []):
    if len(h):
        axes[0].plot(h["epoch"], h["val_AUC"], marker="o", label=name)
        axes[1].plot(h["epoch"], h["val_LogLoss"], marker="o", label=name)
axes[0].set(title="Validation AUC", xlabel="epoch")
axes[1].set(title="Validation log loss", xlabel="epoch")
axes[0].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "learning_curves.png", dpi=120)
plt.show()

## 7. Final test (scored once, after every choice was made on validation)

The confidence intervals resample **students**, not rows. The paired interval for
"AKT − DKT-Q" uses the same resampled students for both models. If it lies entirely above 0, AKT is
better on this data. If it contains 0, the difference is not clear.

In [ ]:
TEST_PRED["AKT"] = first_attempt_fill(predict_akt(AKT_MODEL, "test"), test_rows)
if TRAIN_DKTQ_BASELINE:
    TEST_PRED["DKT-Q"] = first_attempt_fill(predict_dktq(DKTQ_MODEL, "test"), test_rows)

groups = df.loc[test_rows, "user_id"].to_numpy()
uniq, inv = np.unique(groups, return_inverse=True)
rows_of = [np.flatnonzero(inv == k) for k in range(len(uniq))]
rng = np.random.default_rng(SEED)
REPS = 50 if SMOKE else 300
boot = {m: [] for m in TEST_PRED}
for _ in range(REPS):
    pick = np.concatenate([rows_of[k] for k in rng.integers(0, len(uniq), len(uniq))])
    for m in TEST_PRED:
        boot[m].append(roc_auc_score(y_test[pick], TEST_PRED[m][pick]))

rows_out = []
for m in TEST_PRED:
    lo, hi = np.percentile(boot[m], [2.5, 97.5])
    rows_out.append({"Model": m, **metrics(y_test, TEST_PRED[m]), "AUC 95% CI": f"{lo:.3f}-{hi:.3f}"})
test_table = pd.DataFrame(rows_out).set_index("Model").sort_values("LogLoss")
display(test_table.round(4))

diff = None
if TRAIN_DKTQ_BASELINE:
    d = np.array(boot["AKT"]) - np.array(boot["DKT-Q"])
    point = metrics(y_test, TEST_PRED["AKT"])["AUC"] - metrics(y_test, TEST_PRED["DKT-Q"])["AUC"]
    lo, hi = np.percentile(d, [2.5, 97.5])
    diff = {"AUC_AKT_minus_DKTQ": point, "CI95": [lo, hi]}
    verdict = "AKT is better" if lo > 0 else ("DKT-Q is better" if hi < 0 else "no clear difference")
    print(f"AUC(AKT) - AUC(DKT-Q) = {point:+.4f}  (95% CI {lo:+.4f} to {hi:+.4f})  ->  {verdict}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for name, p in TEST_PRED.items():
    bins = np.clip((p * 10).astype(int), 0, 9)
    ok = [b for b in range(10) if (bins == b).sum() > 100]
    ax.plot([p[bins == b].mean() for b in ok], [y_test[bins == b].mean() for b in ok], marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set(xlabel="predicted P(correct)", ylabel="observed correct rate", title="Reliability (test students)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "reliability_test.png", dpi=120)
plt.show()

# MENTORX picks questions near P(correct) = 0.65, so check calibration in that band
for name, p in TEST_PRED.items():
    band = (p >= 0.55) & (p <= 0.75)
    print(f"{name}: band 0.55-0.75 n={band.sum():,} predicted={p[band].mean():.3f} observed={y_test[band].mean():.3f}")

## 8. Save everything

`mentorx_akt/` contains `akt.pt` (weights, config, and the exact skill and question id order the
embeddings use), `dkt_q.pt`, the training histories, `results_akt.json` and the plots. To serve AKT,
MENTORX would need a NumPy port of this forward pass (like `backend/app/ml/dkt.py` for DKT-Q). That is
only worth doing if the test difference above is clearly positive.

In [ ]:
ck = torch.load(ckpt_path(BEST_AKT), map_location="cpu")
ck.update({"config": AKT_RUNS[BEST_AKT]["cfg"], "lr": AKT_RUNS[BEST_AKT]["lr"], "setting": BEST_AKT, "l2_rasch": L2_RASCH, "max_sk": MAX_SK,
           "skill_ids": [int(s) for s in skill_ids], "question_ids": [int(q) for q in train_q],
           "eval_window": EVAL_WINDOW, "seed": SEED})
torch.save(ck, OUT_DIR / "akt.pt")  # the chosen setting, ready for porting

results = {
    "dataset": "FoundationalASSIST (ASSISTments; CC BY-NC 4.0)",
    "split": "by student 70/15/15, seed 42 (same as notebook 02)",
    "smoke_test": SMOKE,
    "best_akt": BEST_AKT, "akt_config": AKT_RUNS[BEST_AKT]["cfg"], "lr": AKT_RUNS[BEST_AKT]["lr"],
    "akt_runs": {m: {**AKT_RUNS[m], "epochs_run": len(AKT_HISTS[m]),
                     "best_epoch": int(AKT_HISTS[m].loc[AKT_HISTS[m]["val_LogLoss"].idxmin(), "epoch"])
                     if len(AKT_HISTS[m]) else None} for m in AKT_RUNS},
    "l2_rasch": L2_RASCH, "train_window": TRAIN_WINDOW, "eval_window": EVAL_WINDOW,
    "val": val_table.round(5).to_dict(orient="index"),
    "test": test_table.round(5).to_dict(orient="index"),
    "paired_auc_difference": diff,
    "akt_epochs": AKT_HIST.to_dict(orient="records") if len(AKT_HIST) else [],
    "environment": {"device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                    "torch": torch.__version__, "python": platform.python_version()},
}
json.dump(results, open(OUT_DIR / "results_akt.json", "w"), indent=2, default=float)

print("=" * 30, "MENTORX AKT RUN SUMMARY", "=" * 30)
print(f"best AKT setting (validation): {BEST_AKT} {AKT_RUNS[BEST_AKT]}")
print(f"device={DEVICE} gpu={results['environment']['gpu']} torch={torch.__version__} "
      f"rows={len(df):,} students={df.user_id.nunique():,} seed={SEED} smoke={SMOKE}")
print("\n--- training log ---")
print(open(OUT_DIR / "train_log.txt").read())
print("--- validation ---")
print(val_table.round(4).to_string())
print("\n--- test (scored once) ---")
print(test_table.round(4).to_string())
if diff:
    print(f"\nAUC(AKT) - AUC(DKT-Q) = {diff['AUC_AKT_minus_DKTQ']:+.4f}  CI95 {diff['CI95'][0]:+.4f} to {diff['CI95'][1]:+.4f}")
print("\nFiles:", sorted(p.name for p in OUT_DIR.iterdir()))
print("=" * 85)